# Tahap 2 — Data Cleaning
### Proyek: Prediksi Customer Churn pada Perusahaan Telekomunikasi

Lanjutan dari `01_business_data_understanding.ipynb`. Tiga temuan dari
Tahap 1 yang jadi fokus pembersihan di sini:

1. `TotalCharges` tersimpan sebagai teks dan punya **11 baris "kosong
   tersembunyi"** (string spasi) — semuanya pelanggan dengan `tenure = 0`.
2. `SeniorCitizen` disimpan sebagai `0`/`1`, tidak konsisten dengan kolom
   biner lain yang memakai `Yes`/`No`.
3. `customerID` perlu dipastikan tidak ada duplikat (di Tahap 1 hasilnya 0
   duplikat, tapi tetap kita cek ulang & tangani di sini secara eksplisit
   supaya notebook ini berdiri sendiri).

Output notebook ini adalah dataset yang sudah dibersihkan, disimpan ke
`data/interim/` (belum final untuk modeling — feature engineering di
Tahap 4 masih akan menambah kolom baru).

## 1. Setup & Load Data Mentah

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 120)

RAW_PATH = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(RAW_PATH)

print(f"Shape awal: {df.shape}")
df.head()

Shape awal: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Membersihkan `TotalCharges`

### 2.1 Konversi ke numerik

`TotalCharges` dibaca sebagai `object` karena ada baris dengan nilai berupa
string kosong (`" "`). `pd.to_numeric(..., errors="coerce")` akan mengubah
baris tersebut jadi `NaN` sehingga bisa ditangani secara eksplisit.

In [2]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

n_missing = df["TotalCharges"].isnull().sum()
print(f"Jumlah TotalCharges yang jadi NaN setelah konversi: {n_missing}")
df.loc[df["TotalCharges"].isnull(), ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

Jumlah TotalCharges yang jadi NaN setelah konversi: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,NaN
753,3115-CZMZD,0,20.25,NaN
936,5709-LVOEQ,0,80.85,NaN
1082,4367-NUYAO,0,25.75,NaN
1340,1371-DWPAZ,0,56.05,NaN
3331,7644-OMVMY,0,19.85,NaN
3826,3213-VVOLG,0,25.35,NaN
4380,2520-SGTTA,0,20.00,NaN
5218,2923-ARZLG,0,19.70,NaN
6670,4075-WKNIU,0,73.35,NaN


### 2.2 Menangani baris kosong

Seperti dikonfirmasi di Tahap 1, seluruh baris yang kosong punya
`tenure = 0` — artinya pelanggan yang baru mendaftar dan belum pernah
ditagih sama sekali. Jadi nilai yang paling masuk akal secara bisnis untuk
`TotalCharges` mereka adalah **0**, bukan rata-rata/median dari pelanggan
lain (yang justru akan menyesatkan karena tidak merepresentasikan kondisi
pelanggan baru).

Alternatif `MonthlyCharges * tenure` juga menghasilkan 0 karena
`tenure = 0`, jadi kedua pendekatan sepakat pada nilai yang sama.

In [3]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

assert df["TotalCharges"].isnull().sum() == 0
print("TotalCharges sekarang numerik dan tidak ada missing value.")
df["TotalCharges"].describe()

TotalCharges sekarang numerik dan tidak ada missing value.


count    7043.000000
mean     2279.734304
std      2266.794470
min         0.000000
25%       398.550000
50%      1394.550000
75%      3786.600000
max      8684.800000
Name: TotalCharges, dtype: float64

## 3. Menyeragamkan `SeniorCitizen`

Kolom ini berisi `0`/`1`, sementara kolom biner lain (`Partner`,
`Dependents`, `PhoneService`, dst.) memakai `Yes`/`No`. Supaya konsisten
dan lebih mudah dibaca saat EDA, `SeniorCitizen` diubah jadi `Yes`/`No`.

In [4]:
print("Sebelum:", df["SeniorCitizen"].unique())

df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

print("Setelah :", df["SeniorCitizen"].unique())
df["SeniorCitizen"].value_counts()

Sebelum: [0 1]
Setelah : <StringArray>
['No', 'Yes']
Length: 2, dtype: str


SeniorCitizen
No     5901
Yes    1142
Name: count, dtype: int64

## 4. Cek & Hapus Duplikasi `customerID`

Dicek ulang secara eksplisit di notebook ini (bukan hanya mengandalkan hasil
Tahap 1), dan baris duplikat (jika ada) dibuang, menyisakan kemunculan
pertama saja.

In [5]:
n_before = len(df)
n_duplicate_ids = df["customerID"].duplicated().sum()
print(f"Jumlah baris sebelum        : {n_before}")
print(f"Jumlah customerID duplikat  : {n_duplicate_ids}")

if n_duplicate_ids > 0:
    df = df.drop_duplicates(subset="customerID", keep="first").reset_index(drop=True)

n_after = len(df)
print(f"Jumlah baris setelah        : {n_after}")
assert df["customerID"].is_unique

Jumlah baris sebelum        : 7043
Jumlah customerID duplikat  : 0
Jumlah baris setelah        : 7043


## 5. Pemeriksaan Akhir

Pastikan tidak ada missing value tersisa dan tipe data sudah sesuai
ekspektasi sebelum data disimpan.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   str    
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [7]:
total_missing = df.isnull().sum().sum()
print(f"Total missing value di seluruh dataset: {total_missing}")
assert total_missing == 0

Total missing value di seluruh dataset: 0


## 6. Simpan Data Bersih

Disimpan ke `data/interim/` (bukan `data/processed/`) karena ini baru hasil
*cleaning*, belum melalui feature engineering (encoding, fitur turunan,
penanganan class imbalance) yang jadi bagian Tahap 4.

In [8]:
OUTPUT_PATH = "../data/interim/telco_customer_churn_cleaned.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Data bersih disimpan ke: {OUTPUT_PATH}")
print(f"Shape akhir: {df.shape}")

Data bersih disimpan ke: ../data/interim/telco_customer_churn_cleaned.csv
Shape akhir: (7043, 21)


## 7. Ringkasan Tahap 2

**Yang sudah dilakukan:**

- `TotalCharges` dikonversi dari teks ke numerik; 11 baris kosong (semua
  `tenure = 0`) diisi dengan `0`.
- `SeniorCitizen` diseragamkan dari `0`/`1` menjadi `No`/`Yes` agar
  konsisten dengan kolom biner lain.
- Duplikasi `customerID` dicek ulang dan ditangani (hasil: tidak ada
  duplikat, jadi jumlah baris tetap 7.043).
- Dataset bersih disimpan ke `data/interim/telco_customer_churn_cleaned.csv`.

**Lanjut ke Tahap 3 — Exploratory Data Analysis (EDA):**

1. Distribusi target `Churn` (persentase Yes vs No).
2. Bandingkan churn rate terhadap `Contract`, `tenure`, `MonthlyCharges`,
   `InternetService`.
3. Buat minimal 4–6 visualisasi (bar chart, histogram, boxplot, correlation
   heatmap).